# Part A. Setup and data.

## Loading everything we will need.
I start with the imports.

In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
                            
import keras
from keras import layers

## Loading the data.
I download the Adult data set from OpenML. The features arrive in `X` as a table, 
and the target in `y` as text, either `>50K` or `<=50K`. 

In [2]:
X, y = fetch_openml("adult", version=2, as_frame=True, return_X_y=True)

print("Rows and columns:", X.shape)
print("Target values:", y.value_counts().to_dict())
print(X.head())

Rows and columns: (48842, 14)
Target values: {'<=50K': 37155, '>50K': 11687}
   age  workclass  fnlwgt     education  education-num      marital-status  \
0   25    Private  226802          11th              7       Never-married   
1   38    Private   89814       HS-grad              9  Married-civ-spouse   
2   28  Local-gov  336951    Assoc-acdm             12  Married-civ-spouse   
3   44    Private  160323  Some-college             10  Married-civ-spouse   
4   18        NaN  103497  Some-college             10       Never-married   

          occupation relationship   race     sex  capital-gain  capital-loss  \
0  Machine-op-inspct    Own-child  Black    Male             0             0   
1    Farming-fishing      Husband  White    Male             0             0   
2    Protective-serv      Husband  White    Male             0             0   
3  Machine-op-inspct      Husband  Black    Male          7688             0   
4                NaN    Own-child  White  Female      

## Encoding the target.
    The target is text with two values. I convert it to numbers so both models
    can use it. LabelEncoder maps the two classes to 0 and 1 in alphabetical 
    order, so <=50K becomes 0 and >50K becomes 1. The higher income class is 
    therefore the positive class, labelled 1.

In [3]:
y = LabelEncoder().fit_transform(y)
print("Positive rate (share earning >50K):", round(y.mean(), 3))

Positive rate (share earning >50K): 0.239


## Train and test split.
    I split the data into a training set and a test set before doing anything 
    else, and I keep the class balance with stratify. Both models are trained
    on the training set and judged on the same held-back test set.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Training rows:", X_train.shape[0])
print("Test rows:    ", X_test.shape[0])  

Training rows: 39073
Test rows:     9769


## Numeric and categorical columns.
    These two kinds of feature need different preprocessing, so I sort the 
    columns into a numeric group and a categorical group. 
    I detect them from the column data types.

In [5]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical features: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


# Part B. The classical model.
    
## Preprocessor with a ColumnTransformer.

### For the NUMERIC columns, 
    I chain two steps in a Pipeline:
        -SimpleImputer to fill missing values (strategy="median")
        -StandardScaler to put the features on the same scale.

In [6]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

###  For the CATEGORICAL columns, 
    I chain two steps in a Pipeline:
      -SimpleImputer to fill missing values (strategy="most_frequent")
      -OneHotEncoder to turn categories into numbers. 
    I set:
      -handle_unknown="ignore", so unseen categories in the test set do not error
      -sparse_output=False, so the result is a plain array the network can use 
      later.

In [7]:
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

Then I combine the two with a ColumnTransformer, applying each pipeline
to its own list of columns (numeric_features and categorical_features).

In [8]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_features),
        ("categ", categorical_pipe, categorical_features),
        ],
        remainder="drop",
)

I combine the preprocessor and a LogisticRegression into one Pipeline,
so that preprocessing is fitted only on the training data inside each fold.
LogisticRegression(max_iter=1000) gives it enough iterations to converge.

In [9]:
pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])
print(pipe)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'fnlwgt',
                                                   'education-num',
                                                   'capital-gain',
                                                   'capital-loss',
                                                   'hours-per-week']),
                                                 ('categ',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy

I run a 5-fold cross-validation on the TRAINING data, scoring by F1 and
using StratifiedKFold(n_splits=5, shuffle=True, random_state=42).

In [ ]:
cv = StratifiedKFold(
    n_splits=5, 
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    pipe,
    X_train,
    y_train,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

print("F1 Score:", cv_scores)
print("Mean F1:", cv_scores.mean().round(4))
print("Standard Deviation:", cv_scores.std().round(4))

I fit the classifier(clf) on the training data, then predict on the test data.
I store the results so I can compare them later. 
Then I draw the confusion matrix for the classical model.

In [ ]:
pipe.fit(
    X_train,
    y_train
)
y_predict = pipe.predict(X_test)

clf_acc = round(accuracy_score(y_test, y_predict), 4)
clf_f1 = round(f1_score(y_test, y_predict), 4)

print("Accuracy:", clf_acc)
print("F1 score:", clf_f1)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_predict,
    display_labels=["<=50K", ">50K"],
    cmap="Blues"
)

plt.title("Confusion Matrix - Logistic Regression")
plt.show(block=False)
plt.pause(1)
plt.close()

# The code above was necessary because it was interfering when running 
# the script from top to bottom.

## Part C. The neural network.

### Before building the neural network, make a prediction.
Do you expect its F1 score to be higher than, lower than, 
or similar to that of logistic regression? Do you expect a large or
a small difference? Briefly explain your reasoning.

MY PREDICTION: I expect the F1 score to be similar to that of logistic
regression, with only a small difference. Logistic regression is already a
strong baseline for this tabular data after numeric scaling and categorical
one-hot encoding. The neural network may learn some nonlinear interactions,
but its improvement is unlikely to be large because the input representation
and the underlying task are well suited to a linear model.

A neural network needs a plain numeric array as input, so I apply the 
preprocessor to turn the mixed table into numbers. I fit the preprocessor 
on the training data only, then transform both sets, so no information 
leaks from the test set.

In [ ]:
X_train_prep = preprocess.fit_transform(X_train)   # fit on training data only
X_test_prep  = preprocess.transform(X_test)        # reuse the same fitting

print("Training shape:", X_train_prep.shape)

### I build a dense network with the Keras Sequential API.
     -an Input layer whose shape is the number of columns in X_train_prep (X_train_prep.shape[1])                                 
     -a Dense hidden layer (try 64 units, relu)
     -a Dropout layer (0.3) to reduce overfitting
     -a second Dense hidden layer (try 32 units, relu)
     -a Dense output layer with 1 unit and a sigmoid activation

In [ ]:
net = keras.Sequential([
    layers.Input(shape=(X_train_prep.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

net.summary()

### I compile net with:
    -optimizer="adam", 
    -loss="binary_crossentropy", 
    -metrics=["accuracy"]
### Then I train it with:
    -an explicit 10% validation set, 
    -an EarlyStopping callback (monitor "val_loss",
    -patience about 3, restore_best_weights=True), 
    -up to about 20 epochs, and 
    -batch_size=128. 
I store the result in history.

In [ ]:
net.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = net.fit(
    X_train_prep, 
    y_train,
    validation_split=0.1,
    callbacks=[early_stopping],
    epochs=20,
    batch_size=128
)

Now I produce predictions and evaluate the network.
Then I compute nn_acc and nn_f1, printing them, and draw the confusion matrix.

In [ ]:
nn_prob = net.predict(X_test_prep).ravel()     # probabilities
nn_pred = (nn_prob > 0.5).astype(int)           # threshold at 0.5

nn_acc = round(accuracy_score(y_test, nn_pred), 4)
nn_f1 = round(f1_score(y_test, nn_pred), 4)

print("Accuracy NN:", nn_acc)
print("F1 Score NN:", nn_f1)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    nn_pred,
    display_labels=["<=50K", ">50K"],
    cmap="Blues"
)

plt.title("Confusion Matrix - Neural Network")
plt.show()

## Part D. The comparison.

Now we answer the central question. We have four numbers: the accuracy and 
F1 of the classical model, and the accuracy and F1 of the network, all 
measured on the same test set

In [ ]:
print("Model Comparison:")
print(
    f"Logistic Regression F1: {clf_f1:.4f}" 
    f"Logistic Regression Accuracy: {clf_acc:.4f}"
)
print(
    f"Neural Network F1: {nn_f1:.4f}" 
    f"Neural Network Accuracy: {nn_acc:.4f}"
)

# Ethical considerations

here are a few ethical risks we have to consider before deploying this model.
Firstly, a model that uses characteristics like sex, race, and country of origin
in order to predict their income bracket is incredibly sensitive to reproducing 
existing inequalities. Ensuring equity and fairness is a priority. For public 
benefit programmes, a false negative may exclude someone who needs support, 
while a false positive may waste resources. Those mistakes have great real-world
cost for people, agencies, and government, defeating also the purpose of the 
programme and wasting funds. In my view, such considerations make a simpler,
more transparent model more preferable to an opaque one when decisions affect people.

# Wrapping up

Overall, the neural network (NN) model outperforms the logistic regression (LR).
As expected, F1 scores across both models were similar (LR F1= 0.6552, NN F1= 0.6667) with 
small diferences. Accuracy scores were similar (LR accuracy= 0.8524, NN accuracy= 0.8560).
The Accuracy and F1 scores of the NN are less stable than the ones of the LR, 
because their results can depend on random initialization and training order. 
Repeated runs are needed to demonstrate this.

While accuracy is a very important metric, we should emphasise that a single number
cannot tell us the whole story and might mislead us into a false sense of confidence
about how well our model is learning and performing. Firstly, accuracy is important 
in the case of a reasonably balanced data set; this dataset is imbalanced. The 
majority class has more examples, it contributes more to the overall loss. The model
might not be performing well on the minority class. F1 is the more informative metric, 
because it balances precision and recall for the positive, minority class (>50K).
However, F1 does not measure fairness across demographic groups. F1 is your main 
comparison metric and accuracy as supporting information. 

We need to understand how the model performs on each class separately. Confusion
matrices allow us to examine exactly that, but also compare how each model performs
and where each model is struggling. Both False Positives(FP) and False Negatives(FN)
are slightly higher in the LR, but both make similar mistakes.
(LR: 962 FP,  NN: 931 FP; LR: 480 FN, NN: 476 FN)

Finally, the neural network was more successful in identifying the higher-income class
(NN: 1407 True Positives vs LR: 1376 True Positives). The network required a few 
more careful steps, due to its greater complexity. On this data set and setting,
the extra complexity of this model resulted in slightly better predictions when 
compared to the simpler model. The Adult dataset is large enough for the neural 
network to learn some nonlinear interactions between features such as education, 
occupation, age, and hours worked. These interactions may explain its slightly 
higher F1 score, although the one-hot encoded tabular structure also makes
 logistic regression a strong and competitive baseline.
